In [0]:
import requests
import pandas as pd
from pprint import pprint
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window



In [0]:
stations = (
    spark.table("hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations")
    .select("stationName")
    .rdd
    .map(lambda r: r["stationName"])
    .collect()
)

stations


['중구',
 '한강대로',
 '종로구',
 '청계천로',
 '종로',
 '용산구',
 '광진구',
 '성동구',
 '강변북로',
 '중랑구',
 '동대문구',
 '홍릉로',
 '성북구',
 '정릉로',
 '도봉구',
 '은평구',
 '서대문구',
 '마포구',
 '신촌로',
 '강서구',
 '공항대로',
 '구로구',
 '영등포구',
 '영등포로',
 '동작구',
 '동작대로 중앙차로',
 '관악구',
 '강남구',
 '서초구',
 '도산대로',
 '강남대로',
 '송파구',
 '강동구',
 '천호대로',
 '금천구',
 '시흥대로',
 '강북구',
 '양천구',
 '노원구',
 '화랑로']

In [0]:
# ✅ 공공데이터포털 API Key
# (URL Decoding Key 사용 권장)
SERVICE_KEY_2 = "qz6MbARyTC9CL2GrNus/xLfgJolMh3LYaY+kT98k6wDcHDbTZ/vRAvw8JGmU9PnK25a7lM/ePpU1Dl7AsKAyXw=="

In [0]:
air_url = "http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty"

rows = []

for station in stations:
    params = {
        "serviceKey": SERVICE_KEY_2,
        "returnType": "json",
        "numOfRows": "100", 
        "pageNo": "1",
        "stationName": station,
        "dataTerm": "DAILY",
        "ver": "1.0"
    }

    r = requests.get(air_url, params=params)

    if r.status_code != 200:
        print(f"❌ 실패: {station}")
        continue

    items = r.json()["response"]["body"]["items"]

    for i in items:
        rows.append({
            "stationName": station,
            "dataTime": i.get("dataTime"),
            "khaiValue": i.get("khaiValue"),
            "khaiGrade": i.get("khaiGrade"),
            "pm10Value": i.get("pm10Value"),
            "pm25Value": i.get("pm25Value")
        })

In [0]:
air_pdf = pd.DataFrame(rows)
air_sdf = spark.createDataFrame(air_pdf)

air_sdf.display()

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value
중구,2026-04-30 10:00,63,2,29,17
중구,2026-04-30 09:00,54,2,29,18
중구,2026-04-30 08:00,60,2,27,15
중구,2026-04-30 07:00,54,2,27,17
중구,2026-04-30 06:00,54,2,26,15
중구,2026-04-30 05:00,54,2,27,17
중구,2026-04-30 04:00,54,2,31,18
중구,2026-04-30 03:00,60,2,32,17
중구,2026-04-30 02:00,54,2,30,17
중구,2026-04-30 01:00,64,2,30,16


In [0]:
display(air_sdf[air_sdf["stationName"]=="중구"])

stationName,dataTime,khaiValue,khaiGrade,pm10Value,pm25Value
중구,2026-04-30 10:00,63,2,29,17
중구,2026-04-30 09:00,54,2,29,18
중구,2026-04-30 08:00,60,2,27,15
중구,2026-04-30 07:00,54,2,27,17
중구,2026-04-30 06:00,54,2,26,15
중구,2026-04-30 05:00,54,2,27,17
중구,2026-04-30 04:00,54,2,31,18
중구,2026-04-30 03:00,60,2,32,17
중구,2026-04-30 02:00,54,2,30,17
중구,2026-04-30 01:00,64,2,30,16


In [0]:
air_sdf.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("hive_metastore.demo_airstatus_bronze.BRZ_seoul_air_quality_hourly")